# Card-name OCR v1 — Kaggle training

Trains the v1 CRNN against the synthetic name-region pool produced by
`scripts/render_name_pool.py`. Mirrors `scripts/train_name_v1.py` but with
Kaggle-compatible paths.

## Prerequisites — upload these as Kaggle datasets first

1. **Source code** — the `moxify_ocr_model` repo (at least `src/`, `scripts/`,
   `configs/`). Default expected slug: `moxify-ocr-source`.
2. **Rendered pool** — `data/synth_names/v1/` (images + labels.jsonl).
   Default expected slug: `moxify-ocr-name-pool-v1`.

Then attach BOTH datasets to this notebook (Add Data → Your datasets) and
**enable GPU + Internet** in the notebook settings.

If your dataset slugs differ, the second cell auto-discovers them by
looking for the marker files (`labels.jsonl`, `src/moxify_ocr/__init__.py`)
anywhere under `/kaggle/input/`. No hand-editing needed in the common case.

In [ ]:
# Diagnostic-first: list what's actually mounted before assuming a layout.
# Kaggle's mount paths vary — sometimes /kaggle/input/<slug>/, sometimes
# /kaggle/input/datasets/<owner>/<slug>/. Always ls first.
!ls -la /kaggle/input/
!find /kaggle/input -maxdepth 3 -type d 2>/dev/null | head -40

In [ ]:
# Auto-discover the source + pool roots by looking for marker files anywhere
# under /kaggle/input/. Robust to slug variations and to Kaggle's varying
# /kaggle/input/<slug>/ vs /kaggle/input/datasets/<owner>/<slug>/ layouts.
import pathlib
INPUT_ROOT = pathlib.Path("/kaggle/input")

src_marker = next(INPUT_ROOT.rglob("src/moxify_ocr/__init__.py"), None)
pool_marker = next(INPUT_ROOT.rglob("labels.jsonl"), None)

assert src_marker is not None, (
    f"could not find any src/moxify_ocr/ under {INPUT_ROOT}. "
    "Did you attach the source dataset? Check the cell above for what's mounted."
)
assert pool_marker is not None, (
    f"could not find any labels.jsonl under {INPUT_ROOT}. "
    "Did you attach the pool dataset?"
)
src_root = src_marker.parent.parent.parent  # __init__.py → moxify_ocr/ → src/ → repo root
pool_root = pool_marker.parent              # labels.jsonl → pool root
print(f"source: {src_root}")
print(f"pool:   {pool_root}")

# Sanity-check that the pool isn't an macOS-zip-wrapped layout
# (e.g. <pool>/images/images/ instead of <pool>/images/). If so, surface it
# now rather than blowing up halfway through training.
images_dir = pool_root / "images"
assert images_dir.is_dir(), f"expected {images_dir} to exist"
if (images_dir / "images").is_dir():
    print(
        "WARN: detected double-nested images/images/ — using inner dir. "
        "This usually means the upload zip wrapped the contents in an extra dir."
    )
    images_dir = images_dir / "images"
# AppleDouble companion files (._XXX) inflate listing counts on macOS-tar
# uploads but don't break readers; filter for the diagnostic count only.
n_imgs = sum(
    1 for f in images_dir.iterdir() if not f.name.startswith("._")
)
n_rows = sum(1 for _ in pool_marker.open())
print(f"  images: {n_imgs:,}")
print(f"  rows:   {n_rows:,}")
assert n_imgs >= n_rows * 0.99, (
    f"only {n_imgs:,} images for {n_rows:,} manifest rows — pool looks incomplete"
)

In [ ]:
# Output dir + override knobs.
OUTPUT_DIR = "/kaggle/working/name_v1"
CONFIG_OVERRIDES = {
    # 'train.epochs': 25,
    # 'train.lr': 5.0e-4,
    # 'data.batch_size': 128,
}

In [ ]:
# Add the source to sys.path so `import moxify_ocr` resolves.
# Kaggle's /kaggle/input/ is read-only, so pip install -e is fragile —
# direct sys.path manipulation is more robust and survives between cells.
import sys
if str(src_root / "src") not in sys.path:
    sys.path.insert(0, str(src_root / "src"))

# Sanity-check imports + GPU.
import tensorflow as tf
print("tf version:", tf.__version__)
print("GPUs detected:", tf.config.list_physical_devices("GPU"))
from moxify_ocr.data.name_alphabet import NAME_ALPHABET
from moxify_ocr.models.name_region import NAME_NUM_CLASSES, NAME_INPUT_SHAPE
print(f"alphabet: {len(NAME_ALPHABET)} chars  /  num_classes: {NAME_NUM_CLASSES}  /  input: {NAME_INPUT_SHAPE}")

In [ ]:
# Install the bits Kaggle's TF kernel doesn't ship with.
%pip install -q editdistance pyyaml

In [ ]:
# Build a Kaggle-flavoured config by reading the repo's name_v1.yaml and
# overriding the data + train paths. Account for the macOS-zip double-
# nesting case (where pool_root/images/images/ is the actual image dir):
# the pool_root we point at has to be the directory that contains both
# images/ AND labels.jsonl, even when images itself is wrapped.
import yaml
from pathlib import Path

base_cfg_path = src_root / "configs" / "name_v1.yaml"
with base_cfg_path.open() as f:
    cfg = yaml.safe_load(f)

cfg["data"]["pool_root"] = str(pool_root)
cfg["train"]["output_dir"] = OUTPUT_DIR

for dotted, value in CONFIG_OVERRIDES.items():
    parts = dotted.split(".")
    node = cfg
    for p in parts[:-1]:
        node = node.setdefault(p, {})
    node[parts[-1]] = value

kaggle_cfg_path = Path("/kaggle/working/name_v1_kaggle.yaml")
with kaggle_cfg_path.open("w") as f:
    yaml.safe_dump(cfg, f)
print("merged config:")
print(yaml.safe_dump(cfg, sort_keys=False))

In [ ]:
# Run training by invoking the existing script's main() directly. We patch
# sys.argv so its argparse picks up the Kaggle-flavoured config without
# spawning a subprocess (keeps stdout in the notebook).
import importlib.util
import sys

spec = importlib.util.spec_from_file_location(
    "train_name_v1", src_root / "scripts" / "train_name_v1.py"
)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

old_argv = sys.argv
sys.argv = ["train_name_v1", "--config", str(kaggle_cfg_path)]
try:
    module.main()
finally:
    sys.argv = old_argv

In [ ]:
# Confirm the trained model landed in /kaggle/working/ so it's downloadable.
import pathlib
for p in sorted(pathlib.Path(OUTPUT_DIR).rglob("*")):
    print(p, p.stat().st_size if p.is_file() else "(dir)")